In [ ]:
import pandas as pd
import numpy as np
import requests

# 🔴 Crypto Risk Monitoring — BTC Anomaly Detection
Detecting anomalous price movements in Bitcoin using z-score analysis.
Data source: CoinGecko API (last 90 days)

## 1. Data Collection
Fetching daily BTC/USD prices from CoinGecko API.

In [ ]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "90",
    "interval": "daily"
}

response = requests.get(url, params=params)
data = response.json()

In [ ]:
df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
print(df.head())    

In [ ]:
df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.drop(columns=['timestamp'])
df = df[df['date'].dt.date < pd.Timestamp.today().date()]
print(df.head())

In [ ]:
df['return'] = df['price'].pct_change() * 100
print(df.head())


## 2. Anomaly Detection
Calculating daily returns and z-scores to flag abnormal price movements.
- Z-score > 2 or < -2 = anomaly
- Based on 90-day rolling mean and standard deviation

In [ ]:
mean = df['return'].mean()
std = df['return'].std()

df['z_score'] = (df['return'] - mean) / std

print(df.head(10))

In [ ]:
df['is_anomaly'] = df['z_score'].abs() > 2

In [1]:
print(df[df['is_anomaly'] == True])

NameError: name 'df' is not defined

## 3. Visualization
Price chart with anomaly flags — red dots mark days with abnormal returns.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['price'],
    mode='lines',
    name='BTC Price',
    line=dict(color='orange')
))

fig.add_trace(go.Scatter(
    x=df[df['is_anomaly'] == True]['date'],
    y=df[df['is_anomaly'] == True]['price'],
    mode='markers',
    name='Anomaly',
    marker=dict(color='red', size=10)
))

fig.show()

## 4. Rolling Volatility
30-day rolling standard deviation of daily returns.
Higher volatility = higher risk exposure for open positions.

In [ ]:
df['volatility'] = df['return'].rolling(window=30).std()
print(df.tail(10))